# CS570 Week 4 Lab: DataFrames, Spark SQL & Data Investigation

**Points: 100** | **Due: See Canvas**

---

### Instructions

1. Run every code cell and keep all outputs visible
2. Write your code in cells marked `# YOUR CODE`
3. Fill in the **Results Sheet** at the bottom with your exact values
4. Export as PDF (File → Print Preview → Save as PDF)
5. Submit the PDF to Canvas

**Grading:** The Results Sheet is your scorecard. Code cells are your evidence.

---

## Part 0: Setup

In [1]:
import time
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

**Task:** Create a SparkSession. You did this in Week 3.

In [2]:
import os, sys

os.environ['JAVA_HOME'] = r"C:\Program Files\Eclipse Adoptium\jdk-17.0.13.11-hotspot"
os.environ['PATH'] = os.environ['JAVA_HOME'] + r"\bin;" + os.environ['PATH']
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

# Stop any stale/dead session before creating a new one
try:
    existing = SparkSession.getActiveSession()
    if existing:
        existing.stop()
except Exception:
    pass

spark = SparkSession.builder \
    .appName("CS570 Week 4 Lab") \
    .master("local[*]") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.ui.enabled", "false") \
    .getOrCreate()

sc = spark.sparkContext
sc.setLogLevel("ERROR")

print(f"Spark version: {spark.version}")
print(f"Default parallelism: {sc.defaultParallelism}")


Spark version: 4.1.1
Default parallelism: 16


In [3]:
# Dataset: spotify.csv (same as Week 3)
# If you don't have it, download from the Week 3 lab instructions
data_path = "spotify.csv"

---
## Part 1: Loading & Schema Validation (15 pts)

In production, you always need to validate that your data loaded correctly.

### 1.1 Load with inferSchema

In [4]:
start = time.time()

df_infer = spark.read \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .csv(data_path)

df_infer.count()  # force evaluation

infer_time = time.time() - start
print(f"inferSchema load time: {infer_time:.2f} seconds")

inferSchema load time: 8.19 seconds


### 1.2 Examine the schema

**Task:** Print the schema and show the first few rows. Report the number of columns and rows.

In [5]:
# YOUR CODE: Print the schema that Spark inferred
df_infer.printSchema()

root
 |-- _c0: integer (nullable = true)
 |-- track_id: string (nullable = true)
 |-- artists: string (nullable = true)
 |-- album_name: string (nullable = true)
 |-- track_name: string (nullable = true)
 |-- popularity: integer (nullable = true)
 |-- duration_ms: integer (nullable = true)
 |-- explicit: boolean (nullable = true)
 |-- danceability: double (nullable = true)
 |-- energy: double (nullable = true)
 |-- key: integer (nullable = true)
 |-- loudness: double (nullable = true)
 |-- mode: integer (nullable = true)
 |-- speechiness: double (nullable = true)
 |-- acousticness: double (nullable = true)
 |-- instrumentalness: double (nullable = true)
 |-- liveness: double (nullable = true)
 |-- valence: double (nullable = true)
 |-- tempo: double (nullable = true)
 |-- time_signature: integer (nullable = true)
 |-- track_genre: string (nullable = true)



In [6]:
# YOUR CODE: Show the first 5 rows
df_infer.show(5)

+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|_c0|            track_id|             artists|          album_name|          track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|  tempo|time_signature|track_genre|
+---+--------------------+--------------------+--------------------+--------------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|  0|5SuOikwiRyPMVoIQD...|         Gen Hoshino|              Comedy|              Comedy|        73|     230666|   false|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715| 87.917|            

In [7]:
# YOUR CODE: How many columns? How many rows?
num_cols = len(df_infer.columns)
num_rows = df_infer.count()
print(f"Columns: {num_cols}")
print(f"Rows: {num_rows}")

Columns: 21
Rows: 114000


**Task:** Look at the columns. Are there any that don't add analytical value? Drop them.

In [8]:
# YOUR CODE: Drop any columns that don't add value
# Reassign to df_infer
df_infer = df_infer.drop("_c0")
print(f"Coulumns after drop: {len(df_infer.columns)}")
print(df_infer.columns)

Coulumns after drop: 20
['track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']


### 1.3 Load with explicit schema

Explicit schemas are faster and catch data problems early.

In [9]:
explicit_schema = StructType([
    StructField("_c0", IntegerType(), True),
    StructField("track_id", StringType(), True),
    StructField("artists", StringType(), True),
    StructField("album_name", StringType(), True),
    StructField("track_name", StringType(), True),
    StructField("popularity", IntegerType(), True),
    StructField("duration_ms", IntegerType(), True),
    StructField("explicit", BooleanType(), True),
    StructField("danceability", DoubleType(), True),
    StructField("energy", DoubleType(), True),
    StructField("key", IntegerType(), True),
    StructField("loudness", DoubleType(), True),
    StructField("mode", IntegerType(), True),
    StructField("speechiness", DoubleType(), True),
    StructField("acousticness", DoubleType(), True),
    StructField("instrumentalness", DoubleType(), True),
    StructField("liveness", DoubleType(), True),
    StructField("valence", DoubleType(), True),
    StructField("tempo", DoubleType(), True),
    StructField("time_signature", IntegerType(), True),
    StructField("track_genre", StringType(), True)
])

start = time.time()

df = spark.read \
    .option("header", "true") \
    .option("quote", '"') \
    .option("escape", '"') \
    .option("multiLine", "true") \
    .schema(explicit_schema) \
    .csv(data_path)

df.count()  # force evaluation

explicit_time = time.time() - start
print(f"Explicit schema load time: {explicit_time:.2f} seconds")
print(f"Speedup: {infer_time / explicit_time:.2f}x")

Explicit schema load time: 0.57 seconds
Speedup: 14.38x


**Task:** Verify row counts match, then drop the same useless column from df.

In [10]:
# YOUR CODE: Verify both loaded the same number of rows
# Then drop the useless column from df as well
print(f"inferSchema rows: {df_infer.count()}")
print(f"Explicit schema rows: {df.count()}")
print(f"Match: {df_infer.count() == df.count()}")

df = df.drop("_c0")
print(f"\nColumns after drop: {len(df.columns)}")

inferSchema rows: 114000
Explicit schema rows: 114000
Match: True

Columns after drop: 20


**→ Results Sheet: R1, R2, R3, R4**

---
## Part 2: Data Integrity Investigation (30 pts)

How many unique records does this dataset actually contain?

### 2.1 Find the null row

**Task:** Find which columns have null values. Display the row(s) with nulls. Remove them.

In [11]:
# YOUR CODE: Count nulls in each column
null_counts = []

for c in df.columns:
    null_expr = count(when(col(c).isNull(), c)).alias(c)

    null_counts.append(null_expr)
df.select(null_counts).show(truncate=False)

+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|track_id|artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo|time_signature|track_genre|
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+--------------+-----------+
|0       |1      |1         |1         |0         |0          |0       |0           |0     |0  |0       |0   |0          |0           |0               |0       |0      |0    |0             |0          |
+--------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-----+-------------

In [12]:
# YOUR CODE: Display the row(s) that have null values
cond1 = col("track_name").isNull()
cond2 = col("artists").isNull()
cond3 = col("track_id").isNull()

combined_condition = cond1 | cond2 | cond3

null_rows = df.filter(combined_condition)
null_rows.show(truncate=False)

+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|track_id              |artists|album_name|track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence|tempo  |time_signature|track_genre|
+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+-------+--------------+-----------+
|1kR4gIb7nGxHPI3D2ifs59|NULL   |NULL      |NULL      |0         |0          |false   |0.501       |0.583 |7  |-9.46   |0   |0.0605     |0.69        |0.00396         |0.0747  |0.734  |138.391|4             |k-pop      |
+----------------------+-------+----------+----------+----------+-----------+--------+------------+------+---+--------+----+

In [13]:
# YOUR CODE: Remove the null row(s), store as df_clean
# Print row count before and after

df_clean = df.dropna()

print(f"Rows before: {df.count()}")
print(f"Rows after: {df_clean.count()}")

Rows before: 114000
Rows after: 113999


**→ Results Sheet: R5, R6**

### 2.2 Examine the artists column

**Task:** Look closely at the `artists` column. What do you notice about tracks with multiple artists?

In [14]:
# YOUR CODE: Show some rows where it looks like multiple artists are involved
# Hint: look for a pattern in how they're stored
df_clean.select("artists", "track_name").filter(col("artists").contains(";")).show(10, truncate=False)

+------------------------------------+---------------------+
|artists                             |track_name           |
+------------------------------------+---------------------+
|Ingrid Michaelson;ZAYN              |To Begin Again       |
|A Great Big World;Christina Aguilera|Say Something        |
|Jason Mraz;Colbie Caillat           |Lucky                |
|Chord Overstreet;Deepend            |Hold On - Remix      |
|Andrew Foy;Renee Foy                |ily (i love you baby)|
|Andrew Foy;Renee Foy                |At My Worst          |
|Jason Mraz;Colbie Caillat           |Lucky                |
|Boyce Avenue;Bea Miller             |Photograph           |
|Boyce Avenue;Jennel Garcia          |Demons               |
|A Great Big World;Christina Aguilera|Say Something        |
+------------------------------------+---------------------+
only showing top 10 rows


**→ Results Sheet: R7 — How are multiple artists stored in the artists column?**

### 2.3 How many unique tracks?

**Task:** Determine how many unique tracks are in the dataset. But first — what does "unique" mean? You decide.

In [15]:
# YOUR CODE: Count unique tracks
# First, write your definition:
# MY DEFINITION: A unique track is defined by it's track _id (the Spotify unique identifier)
#
# Now implement it:
unique_count = df_clean.select("track_id").distinct().count()
print(f"Unique tracks (by track_id): {unique_count}")


Unique tracks (by track_id): 89740


In [16]:
# YOUR CODE: Compare total rows vs unique tracks
total = df_clean.count()
unique = df_clean.select("track_id").distinct().count()
print(f"Total rows: {total}")
print(f"Unique tracks: {unique}")
print(f"Duplicate rows: {total - unique}")

Total rows: 113999
Unique tracks: 89740
Duplicate rows: 24259


In [17]:
# YOUR CODE: Show the distribution — how many tracks appear 1x, 2x, 3x, etc.
track_counts = df_clean.groupBy("track_id").count()
track_counts.groupBy("count").agg(
    count("*").alias("num_tracks")
).orderBy("count").show()

# Max times a single track appears 
max_appearances = track_counts.agg(max("count")).collect()[0][0]
print(f"Max times a single track appears: {max_appearances}")

+-----+----------+
|count|num_tracks|
+-----+----------+
|    1|     73099|
|    2|     11712|
|    3|      2984|
|    4|      1372|
|    5|       431|
|    6|       117|
|    7|        22|
|    8|         2|
|    9|         1|
+-----+----------+

Max times a single track appears: 9


**→ Results Sheet: R8, R9, R10**

### 2.4 Investigate the duplicates

**Task:** Pick a track that appears multiple times. Show all its rows. Determine: do the audio features (energy, danceability) differ? Does the genre differ?

In [18]:
# YOUR CODE: Pick a track that appears 4+ times, show all its rows
# Include: track_id, track_name, artists, track_genre, energy, danceability
frequent_track = df_clean.groupBy("track_id").count().filter(col("count") >=4).first()["track_id"]

df_clean.filter(col("track_id")== frequent_track) \
    .select("track_id", "track_name", "artists", "track_genre", "energy", "danceability") \
    .show(truncate=False)    

+----------------------+----------+------------------------------+-----------+------+------------+
|track_id              |track_name|artists                       |track_genre|energy|danceability|
+----------------------+----------+------------------------------+-----------+------+------------+
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|alt-rock   |0.461 |0.784       |
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|alternative|0.461 |0.784       |
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|latin      |0.461 |0.784       |
|0txb2qyDV7sVBLv2KvtbW2|Bohemio   |Andrés Calamaro;Julio Iglesias|rock       |0.461 |0.784       |
+----------------------+----------+------------------------------+-----------+------+------------+



In [ ]:
# frequent_track = df_clean.groupBy("track_id").count().filter(col("count") >= 4).collect()[2]["track_id"]
# df_clean.filter(col("track_id")== frequent_track) \
#     .select("track_id", "track_name", "artists", "track_genre", "energy", "danceability") \
#     .show(truncate=False) 

+----------------------+----------+----------+-----------+------+------------+
|track_id              |track_name|artists   |track_genre|energy|danceability|
+----------------------+----------+----------+-----------+------+------------+
|3RlsVPIIs5KFhLFhxZ4iDF|Rockstar  |Nickelback|alt-rock   |0.91  |0.616       |
|3RlsVPIIs5KFhLFhxZ4iDF|Rockstar  |Nickelback|alternative|0.91  |0.616       |
|3RlsVPIIs5KFhLFhxZ4iDF|Rockstar  |Nickelback|grunge     |0.91  |0.616       |
|3RlsVPIIs5KFhLFhxZ4iDF|Rockstar  |Nickelback|metal      |0.91  |0.616       |
+----------------------+----------+----------+-----------+------+------------+



**Task:** Prove whether duplicates have identical audio features or different ones.

In [ ]:
# YOUR CODE: How many track_ids have DIFFERENT energy values across duplicates?
# Hint: groupBy track_id, use countDistinct on energy, filter where > 1
diff_energy = df_clean.groupBy("track_id")\
    .agg(count_distinct("energy").alias("distinct_energy"))\
    .filter(col("distinct_energy")>1)\
    .count()
print(f"Track_ids with differering energy across duplicates: {diff_energy}")

Track_ids with differering energy across duplicates: 0


In [23]:
# YOUR CODE: How many track_ids have DIFFERENT genre values across duplicates?
diff_genre = df_clean.groupBy("track_id")\
    .agg(count_distinct("track_genre").alias("distinct_genre"))\
    .filter(col("distinct_genre")>1)\
    .count()
print(f"Track_ids with differering genre across duplicates: {diff_genre}")

Track_ids with differering genre across duplicates: 16299


**→ Results Sheet: R11, R12, R13**

---
## Part 3: DataFrame Operations & SQL (20 pts)

In [33]:
# Register for SQL
df_clean.createOrReplaceTempView("tracks")

In [47]:
spark.sql("select * from tracks").show(3)

+--------------------+--------------------+----------------+----------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+------+--------------+-----------+
|            track_id|             artists|      album_name|      track_name|popularity|duration_ms|explicit|danceability|energy|key|loudness|mode|speechiness|acousticness|instrumentalness|liveness|valence| tempo|time_signature|track_genre|
+--------------------+--------------------+----------------+----------------+----------+-----------+--------+------------+------+---+--------+----+-----------+------------+----------------+--------+-------+------+--------------+-----------+
|5SuOikwiRyPMVoIQD...|         Gen Hoshino|          Comedy|          Comedy|        73|     230666|   false|       0.676| 0.461|  1|  -6.746|   0|      0.143|      0.0322|         1.01E-6|   0.358|  0.715|87.917|             4|   acoustic|
|4qPNDBW1i3p13qLCt...|        Ben Wo

### 3.1 Impact of duplicates on aggregations

**Task:** Create a deduplicated DataFrame based on your definition of unique track. Then calculate average popularity both ways and compare.

In [48]:
# YOUR CODE: Deduplicate based on YOUR definition of unique track
# Hint: dropDuplicates([...]) is easiest

df_dedup = df_clean.dropDuplicates(["track_id"])

print(f"Rows before dedup: {df_clean.count()}")
print(f"Rows after dedup: {df_dedup.count()}")

Rows before dedup: 113999
Rows after dedup: 89740


In [50]:
# YOUR CODE: Calculate average popularity WITH duplicates and WITHOUT
# Report both values rounded to 2 decimals
avg_with_dup = df_clean.agg(round(avg("popularity"), 2)).collect()[0][0]
avg_without_dup =df_dedup.agg(round(avg("popularity"),2)).collect()[0][0]
print(f"Avg popularity WITH duplicates: {avg_with_dup}")
print(f"Avg popularity WITHOUT duplicates: {avg_without_dup}")

Avg popularity WITH duplicates: 33.24
Avg popularity WITHOUT duplicates: 33.2


**→ Results Sheet: R14, R15**

### 3.2 Normalize popularity

**Task:** All audio features are 0.0–1.0. But popularity is 0–100. Add a new column `popularity_norm` that scales it to 0.0–1.0.

In [51]:
# YOUR CODE: Add popularity_norm column using withColumn
# Show 5 rows with track_name, popularity, popularity_norm to verify
df_clean = df_clean.withColumn("popularity_norm", col("popularity") /100.0)
df_clean.select("track_name", "popularity", "popularity_norm").show(5)

+--------------------+----------+---------------+
|          track_name|popularity|popularity_norm|
+--------------------+----------+---------------+
|              Comedy|        73|           0.73|
|    Ghost - Acoustic|        55|           0.55|
|      To Begin Again|        57|           0.57|
|Can't Help Fallin...|        71|           0.71|
|             Hold On|        82|           0.82|
+--------------------+----------+---------------+
only showing top 5 rows


### 3.3 Unique tracks per genre

**Task:** Using Spark SQL, find how many unique tracks each genre has. Show top 5.

In [56]:
# YOUR CODE: SQL query for unique track count per genre
# Remember: count(*) ≠ unique tracks in denormalized data
df_clean.createOrReplaceTempView("tracks")
spark.sql("""
    select track_genre, count(distinct track_id) as unique_tracks
    from tracks
    group by track_genre
    order by unique_tracks desc
""").show(5, truncate=False)

+-----------+-------------+
|track_genre|unique_tracks|
+-----------+-------------+
|sad        |1000         |
|synth-pop  |1000         |
|spanish    |1000         |
|rock-n-roll|1000         |
|swedish    |1000         |
+-----------+-------------+
only showing top 5 rows


**→ Results Sheet: R16**

---
## Part 4: Catalyst Optimizer (15 pts)

### 4.1 Optimize this query

**Task:** Look at this inefficient query. Write an optimized version.

In [58]:
# Inefficient query:
query_a = df_clean \
    .filter(col("popularity") > 50) \
    .filter(col("energy") > 0.5) \
    .filter(col("popularity") > 70) \
    .select("track_name", "popularity", "energy")

In [59]:
# YOUR CODE: Write an optimized version
# Think: redundant filters, order of operations

query_b = df_clean.select("track_name", "popularity", "energy")\
    .select("track_name", "popularity", "energy")\
        .filter((col("popularity")>70) & (col("energy")> 0.5))

### 4.2 Compare execution plans

**Task:** Run explain() on both queries. What do you notice?

In [60]:
# YOUR CODE: Run explain() on both

print("=== Query A (original): ===")
query_a.explain()

print("\n=== Query B (your version): ===")
query_b.explain()

=== Query A (original): ===
== Physical Plan ==
*(1) Project [track_name#161, popularity#162, energy#166]
+- *(1) Filter (((((isnotnull(popularity#162) AND isnotnull(energy#166)) AND atleastnnonnulls(20, track_id#158, artists#159, album_name#160, track_name#161, popularity#162, duration_ms#163, explicit#164, danceability#165, energy#166, key#167, loudness#168, mode#169, speechiness#170, acousticness#171, instrumentalness#172, liveness#173, valence#174, tempo#175, time_signature#176, track_genre#177)) AND (popularity#162 > 50)) AND (energy#166 > 0.5)) AND (popularity#162 > 70))
   +- FileScan csv [track_id#158,artists#159,album_name#160,track_name#161,popularity#162,duration_ms#163,explicit#164,danceability#165,energy#166,key#167,loudness#168,mode#169,speechiness#170,acousticness#171,instrumentalness#172,liveness#173,valence#174,tempo#175,time_signature#176,track_genre#177] Batched: false, DataFilters: [isnotnull(popularity#162), isnotnull(energy#166), atleastnnonnulls(20, track_id#158,

In [61]:
# YOUR CODE: Verify both return the same number of rows
print(f"Query A rows: {query_a.count()}")
print(f"Query B rows: {query_b.count()}")
print(f"Match: {query_a.count() == query_b.count()}")

Query A rows: 3932
Query B rows: 3932
Match: True


**→ Results Sheet: R17, R18**

---
## Part 5: Pipeline Challenge (20 pts)

**Task:** Build a pipeline to answer this question:

> Which individual artists appear in at least 5 different genres AND have an average track popularity above 50?

**Important:** Remember what you discovered about the `artists` column in Part 2. A track like "Artist A;Artist B;Artist C" should count as appearances for A, B, and C separately.

Requirements:
- Start from df_clean
- Parse the artists column properly
- Show: artist, genre_count, avg_popularity (rounded to 2 decimals)
- Order by genre_count descending, then by avg_popularity descending
- Show top 10

*Hint: Look up `split()` and `explode()` functions.*

In [77]:
# YOUR CODE: Build your pipeline
df_artists = df_clean.withColumn("artist", explode(split(col("artists"), ";")))
df_artists = df_artists.withColumn("artist", trim(col("artist")))

results = df_artists.groupBy("artist").agg(
    count_distinct("track_genre").alias("genre_count"),
    round(avg("popularity"), 2).alias("avg_popularity")
)

results = results.filter((col("genre_count")>=5)&(col("avg_popularity")>50))

results = results.orderBy(desc("genre_count"), desc("avg_popularity"))

In [79]:
# YOUR CODE: Show the result
results.show(10, truncate=False)

+--------------+-----------+--------------+
|artist        |genre_count|avg_popularity|
+--------------+-----------+--------------+
|Halsey        |13         |71.47         |
|blackbear     |13         |62.56         |
|Akon          |13         |50.48         |
|Khalid        |12         |64.35         |
|Shreya Ghoshal|12         |57.38         |
|French Montana|12         |57.05         |
|Sia           |12         |50.37         |
|Alan Walker   |11         |62.56         |
|Nicki Minaj   |11         |62.12         |
|Kygo          |11         |61.94         |
+--------------+-----------+--------------+
only showing top 10 rows


**→ Results Sheet: R19, R20**

---
## Part 6: Cleanup

In [80]:
spark.stop()
print("Spark stopped")

Spark stopped


---

# SUBMISSION INSTRUCTIONS

**You must submit TWO files to Canvas:**

1. **PDF** — Your completed notebook exported as PDF (File → Print Preview → Save as PDF)

2. **JSON** — A file named `W4_LastName_FirstName.json` containing your answers

**JSON Format — copy this template into a text file and fill in your values:**

```json
{
    "student": "LastName_FirstName",
    "R1": ,
    "R2": ,
    "R3": ,
    "R4": ,
    "R5": "",
    "R6": ,
    "R7": "",
    "R8": ,
    "R9": "",
    "R10": ,
    "R11": "",
    "R12": "",
    "R13": "",
    "R14": ,
    "R15": ,
    "R16": "",
    "R17": "",
    "R18": "",
    "R19": "",
    "R20": 
}
```

**Rules:**
- Numbers: no quotes (e.g., `"R4": 20`)
- Text: use quotes (e.g., `"R5": "track_name, artists"`)
- Decimals: round to 2 places where specified
- File must be valid JSON — use [jsonlint.com](https://jsonlint.com) to check before submitting

---

# Grading Rubric

| # | Question | Your Answer | Points |
|---|----------|-------------|--------|
| R1 | inferSchema load time (seconds) | | 3 |
| R2 | Explicit schema load time (seconds) | | 3 |
| R3 | Speedup ratio (R1 / R2) | | 4 |
| R4 | Number of columns (after dropping useless one) | | 5 |
| R5 | Which columns have nulls? | | 3 |
| R6 | Row count after removing nulls | | 3 |
| R7 | How are multiple artists stored? (describe the format) | | 4 |
| R8 | Your unique track count | | 4 |
| R9 | Your definition of "unique track" (one sentence) | | 5 |
| R10 | Max times a single track appears | | 4 |
| R11 | Track_ids with differing energy across duplicates | | 4 |
| R12 | Track_ids with differing genre across duplicates | | 4 |
| R13 | Why do duplicates exist? (one sentence) | | 5 |
| R14 | Avg popularity WITH duplicates (rounded to 2) | | 5 |
| R15 | Avg popularity WITHOUT duplicates (rounded to 2) | | 5 |
| R16 | Genre with most unique tracks + that count | | 6 |
| R17 | Filters in Query A's physical plan | | 6 |
| R18 | Are the plans identical? (yes/no + why) | | 7 |
| R19 | Artist appearing in most genres (meeting criteria) | | 10 |
| R20 | That artist's avg popularity (rounded to 2 decimals) | | 10 |
| | | **TOTAL** | **100** |